In [4]:
#loading the data
import numpy as np
import pandas as pd

In [7]:
df = pd.read_csv(r'/Users/shreeyadewangan/Downloads/ecommerce_clickstream_transactions.csv')

In [8]:
df.head()

,UserID,SessionID,Timestamp,EventType,ProductID,Amount,Outcome
0,1,1,2024-07-07 18:00:26.959902,page_view,NaN,NaN,NaN
1,1,1,2024-03-05 22:01:00.072000,page_view,NaN,NaN,NaN
2,1,1,2024-03-23 22:08:10.568453,product_view,prod_8199,NaN,NaN
3,1,1,2024-03-12 00:32:05.495638,add_to_cart,prod_4112,NaN,NaN
4,1,1,2024-02-25 22:43:01.318876,add_to_cart,prod_3354,NaN,NaN


In [9]:
#Size of dataset
df.shape

(74817, 7)

In [10]:
#Column names
df.columns

Index(['UserID', 'SessionID', 'Timestamp', 'EventType', 'ProductID', 'Amount',
       'Outcome'],
      dtype='object')

In [11]:
#Dataset info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74817 entries, 0 to 74816
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   UserID     74817 non-null  int64  
 1   SessionID  74817 non-null  int64  
 2   Timestamp  74817 non-null  object 
 3   EventType  74817 non-null  object 
 4   ProductID  32113 non-null  object 
 5   Amount     10682 non-null  float64
 6   Outcome    10682 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 4.0+ MB


In [12]:
#missing values
df.isna().sum()

UserID           0
SessionID        0
Timestamp        0
EventType        0
ProductID    42704
Amount       64135
Outcome      64135
dtype: int64

In [18]:
#Event distribution
df['EventType'].value_counts()

EventType
page_view       10819
add_to_cart     10735
product_view    10696
logout          10685
purchase        10682
click           10632
login           10568
Name: count, dtype: int64

In [33]:
#Unique users per event
df.groupby('EventType')['UserID'].nunique()

EventType
add_to_cart     1000
click           1000
login           1000
logout          1000
page_view       1000
product_view    1000
purchase        1000
Name: UserID, dtype: int64

In [28]:
#purchase indicator
df['is_purchase'] = df['EventType'] == 'purchase'
df['is_purchase']

0        False
1        False
2        False
3        False
4        False
         ...  
74812     True
74813    False
74814    False
74815     True
74816    False
Name: is_purchase, Length: 74817, dtype: bool

In [30]:
#Check
df[['EventType','is_purchase']].head(10)

,EventType,is_purchase
0,page_view,False
1,page_view,False
2,product_view,False
3,add_to_cart,False
4,add_to_cart,False
5,page_view,False
6,add_to_cart,False
7,login,False
8,click,False
9,page_view,False


In [34]:
#events per session 
df.groupby('SessionID').size().describe()

count      10.000000
mean     7481.700000
std        56.186297
min      7420.000000
25%      7432.000000
50%      7471.000000
75%      7523.500000
max      7580.000000
dtype: float64

In [35]:
#sessions lead to purchase 
sessions_with_purchase = df[df['is_purchase']]['SessionID'].nunique()
total_sessions = df['SessionID'].nunique()
sessions_with_purchase, total_sessions

(10, 10)

In [36]:
#Session conversion rate 
session_conversion_rate = sessions_with_purchase / total_sessions
session_conversion_rate 

1.0

In [37]:
#Number of events per user
df.groupby('UserID').size().describe()

count    1000.00000
mean       74.81700
std         5.47389
min        58.00000
25%        71.00000
50%        75.00000
75%        79.00000
max        94.00000
dtype: float64

In [38]:
#Number of sessions per user
df.groupby('UserID')['SessionID'].nunique().describe()

count    1000.0
mean       10.0
std         0.0
min        10.0
25%        10.0
50%        10.0
75%        10.0
max        10.0
Name: SessionID, dtype: float64

In [39]:
#Purchases per user
df[df['is_purchase']].groupby('UserID').size().describe()

count    1000.000000
mean       10.682000
std         3.158933
min         3.000000
25%         8.000000
50%        10.000000
75%        13.000000
max        22.000000
dtype: float64

In [40]:
#user-level table
user_df = df.groupby('UserID').agg(
    total_events=('EventType', 'count'),
    total_sessions=('SessionID', 'nunique'),
    total_purchases=('is_purchase', 'sum')
).reset_index()

user_df.head()

,UserID,total_events,total_sessions,total_purchases
0,1,82,10,8
1,2,73,10,13
2,3,64,10,6
3,4,83,10,9
4,5,84,10,7


In [43]:
#user segments
def segment_user(row):
    if row['total_purchases'] >= 3:
        return 'Power Buyer'
    elif row['total_purchases'] >= 1:
        return 'Buyer'
    else:
        return 'Browser'

user_df['user_segment'] = user_df.apply(segment_user, axis=1)
user_df['user_segment']

0      Power Buyer
1      Power Buyer
2      Power Buyer
3      Power Buyer
4      Power Buyer
          ...     
995    Power Buyer
996    Power Buyer
997    Power Buyer
998    Power Buyer
999    Power Buyer
Name: user_segment, Length: 1000, dtype: object

In [44]:
#Segment distribution
user_df['user_segment'].value_counts()

user_segment
Power Buyer    1000
Name: count, dtype: int64